# The 0.96201 Consensus

Public ROC AUC: **0.96201**, verified on this account. A compact rank consensus of public predictions. Source files are hash-checked; shared ancestors are not counted twice. This public-only experiment has no OOF validation claim.

Credits: [Daoviet](https://www.kaggle.com/code/daoviet/s6e10-on-the-journey-to-your-first-top-100), [KRISH RABARI](https://www.kaggle.com/code/rabari9999/s6e10-lb-0-96201-rank-6), [goodpjw2008](https://www.kaggle.com/code/goodpjw2008/s6e10-meta-stack-on-public-oof-lb-0-96197), [F. A. Nina 4](https://www.kaggle.com/code/nina2025/ps-s6e10-h-blend-4), [F. A. Nina 5](https://www.kaggle.com/code/nina2025/ps-s6e10-h-blend-5), [Mayank Raaz](https://www.kaggle.com/code/mayankraaz/s6e10-lb-surface-ascent-v2-0-96196), and the public OOF model authors credited upstream.

In [1]:
from pathlib import Path
import hashlib
import numpy as np
import pandas as pd
from scipy.stats import rankdata

SOURCES = [
    ('daoviet/s6e10-on-the-journey-to-your-first-top-100', 0.8, '8883773ff30ba9a71581657b9daf881ac45652031b6e3ee565844dbcff35797a'),
    ('goodpjw2008/s6e10-meta-stack-on-public-oof-lb-0-96197', 0.15, 'a6f723d0d5aa189e9d88a304414c2f7e7a918cb85b540f4ca5abb260144a75ad'),
    ('nina2025/ps-s6e10-h-blend-4', 0.025, '4aa1f673a6774d3207cade30f59e75fe40f1b11c0745fbad23dfb50cbf8533e9'),
    ('nina2025/ps-s6e10-h-blend-5', 0.025, '07dafbbfba3e2bf696306c333c246c17d6c8c73ec038c7e170ae10ef81136aec'),
]
EXPECTED = 'c2f419d804ccdd808456c0e7cd6a10386358ae51a451b16509f82fa88a0afa4c'
root = Path('/kaggle/input')
sample = next(p for p in root.rglob('sample_submission.csv') if 'playground-series-s6e10' in str(p))
ids = pd.read_csv(sample).id.to_numpy()

def percentile(values):
    return (rankdata(values) - 0.5) / len(values)

prediction = np.zeros(len(ids))
for ref, weight, digest in SOURCES:
    slug = ref.split('/')[1]
    paths = [p for p in root.rglob('submission.csv') if slug in str(p)]
    paths = [p for p in paths if hashlib.sha256(p.read_bytes()).hexdigest() == digest]
    assert len(paths) == 1, f'{ref}: source changed'
    data = pd.read_csv(paths[0])
    assert data.id.is_unique and set(data.id) == set(ids)
    values = data.set_index('id').loc[ids].satisfaction.to_numpy()
    assert np.isfinite(values).all() and ((values >= 0) & (values <= 1)).all()
    prediction += weight * percentile(values)

assert abs(sum(s[1] for s in SOURCES) - 1) < 1e-12
submission = pd.DataFrame({'id': ids, 'satisfaction': percentile(prediction)})
submission.to_csv('submission.csv', index=False, float_format='%.12f')
assert hashlib.sha256(Path('submission.csv').read_bytes()).hexdigest() == EXPECTED
print(f'{len(submission):,} rows. Exact scored prediction reproduced.')
submission.head()


299,844 rows. Exact scored prediction reproduced.


,id,satisfaction
0,699635,0.531685
1,699636,0.584135
2,699637,0.655556
3,699638,0.462099
4,699639,0.337927
